In [ ]:
!pip install scikeras==0.13.0 scikit-learn==1.5.2 -q

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from scikeras.wrappers import KerasClassifier
import pandas as pd
import time

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("Train images shape:", X_train_full.shape)
print("Train labels shape:", y_train_full.shape)
print("Test images shape:", X_test.shape)
print("Test labels shape:", y_test.shape)

In [ ]:
# Sample images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train_full[i], cmap='gray')
    plt.title(class_names[y_train_full[i]])
    plt.axis('off')
plt.suptitle("Sample Fashion-MNIST Images")
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

# Class distribution
unique, counts = np.unique(y_train_full, return_counts=True)
plt.figure(figsize=(8, 5))
plt.bar([class_names[i] for i in unique], counts, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Number of samples")
plt.title("Class Distribution (Training Set)")
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
print("Before preprocessing:")
print("X_train_full shape:", X_train_full.shape)
print("X_test shape:", X_test.shape)

# Flatten
X_train_flat = X_train_full.reshape(X_train_full.shape[0], -1).astype('float32')
X_test_flat = X_test.reshape(X_test.shape[0], -1).astype('float32')

# Normalize to [0,1]
X_train_flat /= 255.0
X_test_flat /= 255.0

# One-hot encode labels
y_train_oh = keras.utils.to_categorical(y_train_full, num_classes=10)
y_test_oh = keras.utils.to_categorical(y_test, num_classes=10)

print("\nAfter preprocessing:")
print("X_train_flat shape:", X_train_flat.shape)
print("X_test_flat shape:", X_test_flat.shape)
print("y_train_oh shape:", y_train_oh.shape)
print("y_test_oh shape:", y_test_oh.shape)

# Train/validation split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_flat, y_train_oh, test_size=0.1, random_state=42, stratify=y_train_full
)
print("\nTrain split:", X_train.shape, "Val split:", X_val.shape)

In [ ]:
def build_baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

baseline_model = build_baseline_model()
baseline_model.summary()

In [ ]:
baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

start_time = time.time()
history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)
baseline_train_time = time.time() - start_time
print(f"\nBaseline training time: {baseline_train_time:.2f} seconds")

In [ ]:
# Training vs Validation Accuracy
plt.figure(figsize=(7, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('accuracy_curve.png', dpi=150)
plt.show()

# Training vs Validation Loss
plt.figure(figsize=(7, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()

In [ ]:
y_pred_probs = baseline_model.predict(X_test_flat)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test  # original integer labels

baseline_acc = accuracy_score(y_true, y_pred)
baseline_prec = precision_score(y_true, y_pred, average='macro')
baseline_rec = recall_score(y_true, y_pred, average='macro')
baseline_f1 = f1_score(y_true, y_pred, average='macro')

print(f"Baseline Test Accuracy:  {baseline_acc:.4f}")
print(f"Baseline Precision:      {baseline_prec:.4f}")
print(f"Baseline Recall:         {baseline_rec:.4f}")
print(f"Baseline F1-score:       {baseline_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix - Baseline Model')
plt.colorbar()
tick_marks = np.arange(10)
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                  color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix_baseline.png', dpi=150)
plt.show()

In [ ]:
def build_model_for_search(hidden_layers=2, hidden_neurons=64, learning_rate=0.001,
                            optimizer_name='adam', activation='relu', dropout_rate=0.0,
                            meta=None):
    model = keras.Sequential()
    model.add(layers.Input(shape=(784,)))
    for _ in range(hidden_layers):
        model.add(layers.Dense(hidden_neurons, activation=activation))
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation='softmax'))

    if optimizer_name == 'adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
param_dist = {
    'model__hidden_layers': [1, 2, 3],
    'model__hidden_neurons': [32, 64, 128, 256],
    'model__learning_rate': [0.1, 0.01, 0.001],
    'model__optimizer_name': ['sgd', 'adam', 'rmsprop'],
    'model__activation': ['relu', 'tanh', 'sigmoid'],
    'model__dropout_rate': [0.0, 0.2, 0.5],
    'batch_size': [16, 32, 64, 128],
    'epochs': [10, 20, 30],
}

clf = KerasClassifier(
    model=build_model_for_search,
    verbose=0
)

# Use a subset of training data for search to keep runtime manageable
subset_size = 10000
idx = np.random.choice(X_train.shape[0], subset_size, replace=False)
X_search = X_train[idx]
y_search = y_train[idx]

random_search = RandomizedSearchCV(
    estimator=clf,
    param_distributions=param_dist,
    n_iter=25,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=1  # keep at 1 for GPU/Keras stability in Colab
)

In [ ]:
start_time = time.time()
random_search.fit(X_search, y_search)
search_time = time.time() - start_time

print(f"\nSearch completed in {search_time/60:.2f} minutes")
print("\nBest Parameters:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV Accuracy: {random_search.best_score_:.4f}")

In [ ]:
results_df = pd.DataFrame(random_search.cv_results_)
results_df_sorted = results_df.sort_values('mean_test_score', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(range(len(results_df_sorted)), results_df_sorted['mean_test_score'], color='teal')
plt.xlabel('Mean CV Accuracy')
plt.ylabel('Hyperparameter Combination (rank)')
plt.title('Hyperparameter Search Results (25 random combinations)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('hp_search_results.png', dpi=150)
plt.show()

results_df_sorted[['params', 'mean_test_score', 'std_test_score']].head(10)

In [ ]:
best_params = random_search.best_params_

optimized_model = build_model_for_search(
    hidden_layers=best_params['model__hidden_layers'],
    hidden_neurons=best_params['model__hidden_neurons'],
    learning_rate=best_params['model__learning_rate'],
    optimizer_name=best_params['model__optimizer_name'],
    activation=best_params['model__activation'],
    dropout_rate=best_params['model__dropout_rate'],
)
optimized_model.summary()

start_time = time.time()
opt_history = optimized_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    verbose=1
)
optimized_train_time = time.time() - start_time
print(f"\nOptimized model training time: {optimized_train_time:.2f} seconds")

In [ ]:
y_pred_probs_opt = optimized_model.predict(X_test_flat)
y_pred_opt = np.argmax(y_pred_probs_opt, axis=1)

opt_acc = accuracy_score(y_true, y_pred_opt)
opt_prec = precision_score(y_true, y_pred_opt, average='macro')
opt_rec = recall_score(y_true, y_pred_opt, average='macro')
opt_f1 = f1_score(y_true, y_pred_opt, average='macro')

print(f"Optimized Test Accuracy:  {opt_acc:.4f}")
print(f"Optimized Precision:      {opt_prec:.4f}")
print(f"Optimized Recall:         {opt_rec:.4f}")
print(f"Optimized F1-score:       {opt_f1:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred_opt, target_names=class_names))

# Confusion Matrix - Optimized
cm_opt = confusion_matrix(y_true, y_pred_opt)
plt.figure(figsize=(8, 7))
plt.imshow(cm_opt, cmap='Greens')
plt.title('Confusion Matrix - Optimized Model')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm_opt[i, j], ha='center', va='center',
                  color='white' if cm_opt[i, j] > cm_opt.max()/2 else 'black')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix_optimized.png', dpi=150)
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'Training Time (s)'],
    'Baseline': [baseline_acc, baseline_prec, baseline_rec, baseline_f1, baseline_train_time],
    'Optimized': [opt_acc, opt_prec, opt_rec, opt_f1, optimized_train_time]
})
print(comparison_df)

# Bar chart comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
baseline_vals = [baseline_acc, baseline_prec, baseline_rec, baseline_f1]
opt_vals = [opt_acc, opt_prec, opt_rec, opt_f1]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(8, 6))
plt.bar(x - width/2, baseline_vals, width, label='Baseline', color='salmon')
plt.bar(x + width/2, opt_vals, width, label='Optimized', color='seagreen')
plt.xticks(x, metrics)
plt.ylabel('Score')
plt.title('Baseline vs Optimized Model Performance')
plt.legend()
plt.tight_layout()
plt.savefig('baseline_vs_optimized.png', dpi=150)
plt.show()